In [ ]:
# IMPORTANT: Due to Windows dynamic library / runtime state collision torch must be imported before mlrun
import torch
import mlrun
# Loads AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and MLRUN_AWS_ROLE_ARN
from dotenv import load_dotenv
load_dotenv() 

from pathlib import Path
from datetime import datetime

artifact_path = Path.cwd().parent
artifact_path = str(artifact_path.as_posix()) # convert windows path to unix path
artifact_path = "file://" + artifact_path
p = mlrun.set_environment("http://localhost:8080", artifact_path=artifact_path)

project = mlrun.load_project(name='finetune-legal-extractor', context="../") # project yaml must be in this directory
# Verify it loaded correctly by checking its status or printing the config
# print(project.to_yaml())

In [ ]:
"""
97082ffd06f34c8d9807a631585f2250
de841f243f9644119e4f093a39518dd4
"""
UID = "de841f243f9644119e4f093a39518dd4"

version = "0.1.2"

run_obj = project.run_function(
    function="register-new-model",    # use the name in the register.ipynb file
    params={
        "experiment_run_uid":UID,
        "version":version
    },
    local=True,
    watch=True,
    verbose=True
)

In [ ]:
import mlrun
from mlrun.model import RunObject

UID = "97082ffd06f34c8d9807a631585f2250"

# Initialize the MLRun DB client
db = mlrun.get_run_db()
run_dict = db.read_run(uid=UID, project="finetune-legal-extractor")
# Convert the dictionary to a RunObject for easier API access
run = RunObject.from_dict(run_dict)

# Get run outputs
run_parameters = run_dict['spec']['parameters']
run_metrics = run_dict['status']['results']
output = run.outputs['return'] # this is what was returned 

print(run_parameters)
print(run_metrics)
print(output)

In [ ]:
# Pass in model_id, commit, hyperparameters, performance metrics

version = datetime.now().strftime("%Y%m%d_%H%M")
hyperparams = run_parameters

m = project.log_model(key="Hermes-4-14B-legal-extractor-adapter",
                  tag=version,
                  metrics=run_metrics,
                  parameters=run_parameters,
                  framework="Hugging Face model",
                  model_url="https://huggingface.co/JerroldK/H4-14b-contract-extractor-adapter",
                  labels={"model": "Hermes-4-14B"},
                  upload=False
)

In [ ]:
m.labels

In [ ]:
model = project.get_artifact(key="Hermes-4-14B-legal-extractor-adapter",
                             tag='latest')
print(model.tag)
print(model.labels)
print(model.model_url)
print(model.metrics)
print(model.parameters)

In [ ]:
# from mlrun.artifacts import ModelArtifact

# # Create a remote model artifact 
# remote_model = ModelArtifact(
#     model_url="https://huggingface.co/JerroldK/Hermes-4-14B-contract-extractor"
# )

# # You can attach the commit hash or revision to the artifact metadata
# remote_model.labels = {"commit_hash": "75875f970c359f89ad9e7d4dc86bf3c075c73c31"}

# # Log it to the project
# project.log_artifact(remote_model, key="Hermes-4-14B-contract-extractor")

In [ ]:
project.save(store=True)